In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os, pickle
d = "/content/drive/MyDrive/radar-trafico/data/raw/session2_center"

print(os.path.getsize(f"{d}/video.avi") == 9508744968)

with open(f"{d}/gt_data.pkl", "rb") as f:

    gt = pickle.load(f, encoding="latin1")

In [ ]:
import pickle

with open("../data/gt_data.pkl", "rb") as f:
    gt = pickle.load(f, encoding="latin1")

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import numpy as np
import pandas as pd
import cv2
from IPython.display import Video, display

def coche_por_id(gt, carId):
    for c in gt["cars"]:
        if c["carId"] == carId:
            return c
    raise ValueError(f"No existe el carId {carId}")

def tiempoCoche(intersections):         # tiempo que tarda en cruzar las lineas 0 y 4 del dataset

    assert intersections[0]["measurementLineId"] == 4

    tiempos = []
    for intersection in intersections:
        tiempos.append(intersection["videoTime"])

    linea0 = tiempos[1]
    linea4 = tiempos[0]


    return linea4, linea0

def leer_frames(rutaVideo, tInicio, tFinal):
    cap = cv2.VideoCapture(rutaVideo)
    assert cap.isOpened()

    cap.set(cv2.CAP_PROP_POS_MSEC, tInicio * 1000)

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
        if t > tFinal:
            break
        yield round(t * 50), t, frame

    cap.release()


def detectar_clip(ruta, tInicio, tFinal, mask, clases=(2, 3, 5, 7)):
    filas = []

    modelo = YOLO("yolo11n.pt")

    for idx, t, frame in leer_frames(ruta, tInicio, tFinal):
        res = modelo.track(frame, persist=True, tracker="bytetrack.yaml",
                   classes=list(clases), verbose=False)[0]
        cajas = res.boxes.xyxy.cpu().numpy()
        confianza = res.boxes.conf.cpu(). numpy()
        clase = res.boxes.cls.cpu().numpy().astype(int)

        if res.boxes.id is None:
            ids = np.full(len(cajas), -1)
        else:
            ids = res.boxes.id.cpu().numpy().astype(int)

        for (x1, y1, x2, y2), conf, cls, tids in zip(cajas, confianza, clase, ids):
            px = min(int((x1 + x2) / 2), mask.shape[1] - 1)
            py = min(int(y2), mask.shape[0] - 1)

            if mask[py, px] > 0:
                filas.append({
                    "frame": idx, "t": t,
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "conf": conf, "cls": cls,
                    "id" : tids,
                })

    return pd.DataFrame(filas)

def dibujar_df(ruta, tInicio, tFinal, df, salida="detecciones.mp4"):
        writer = None
        for idx, t, frame in leer_frames(ruta, tInicio, tFinal):

            for _, f in df[df["frame"] == idx].iterrows():
                x1, y1, x2, y2 = int(f.x1), int(f.y1), int(f.x2), int(f.y2)
                color = COLORES.get(int(f.cls), (255, 255, 255))
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.circle(frame, ((x1 + x2) // 2, y2), 6, color, -1)
                cv2.putText(frame, f"id {int(f.id)} {int(f.cls)} {f.conf:.2f} ", (x1, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

            n = (df["frame"] == idx).sum()
            cv2.putText(frame, f"t={t:.2f}s  cajas={n}", (30, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)

            if writer is None:
                h, w = frame.shape[:2]
                writer = cv2.VideoWriter(salida, cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))
                assert writer.isOpened()
            writer.write(frame)
        writer.release()

In [ ]:
COLORES = {2: (0, 255, 0), 3: (255, 0, 255), 5: (0, 165, 255), 7: (0, 0, 255)}  # BGR: coche verde, camión rojo

CLIPS_TRACKING = {
    "simple" : {
        "carId" : 72,
        "margen" : 0.5,
        "nota" : "turismo solo, carril izquierdo, sin mas coches"
    },

    "paralelo" : {
        "carId" : 350,
        "margen" : 0.5,
        "nota" : "varios coches en escena, en ambos carriles"
    },

    "adelantamiento" : {
        "carId" : 65,
        "margen" : 0.5,
        "nota" : "varios coches en ambos carriles, coche adelantando camion por izquierda"
    }
}

ruta = "/content/drive/MyDrive/radar-trafico/data/raw/session2_center/video.avi"
mask = cv2.imread(f"{d}/video_mask.png", cv2.IMREAD_GRAYSCALE)


In [ ]:
modelo = YOLO("yolo11n.pt")

car = coche_por_id(gt, CLIPS_TRACKING["simple"]["carId"])
tInicio, tFinal = tiempoCoche(car["intersections"])
df = detectar_clip(ruta, tInicio - 1.0, tFinal+ 0.25, mask)

dibujar_df(ruta, tInicio - 1.0, tFinal + 0.25, df)

!ffmpeg -y -loglevel error -i detecciones.mp4 -c:v libx264 detecciones_h264.mp4
display(Video("detecciones_h264.mp4", embed=True))

In [ ]:
xs = np.arange(1920)
a, b, c = gt["measurementLines"][4]
Y_LINEA_4 = -(a * xs + c) / b
a, b, c = gt["measurementLines"][0]
Y_LINEA_0 = -(a * xs + c) / b

def tiempo_entre_lineas(tray, ln4=Y_LINEA_4, ln0=Y_LINEA_0):

    t4 = t0 = None

    primera = tray.iloc[0]
    if primera.y2 >= ln4[int((primera.x1 + primera.x2) / 2)]:
        return None

    for fila in tray.itertuples():
        x = int((fila.x1 + fila.x2) / 2)

        if t4 is None and fila.y2 >= ln4[x]:
            t4 = fila.t
        if t0 is None and fila.y2 >= ln0[x]:
            t0 = fila.t

    if t4 is None or t0 is None:
        return None

    return (t0, t4)


In [ ]:
tray = df[df["id"] == df["id"].mode()[0]].sort_values("frame")
t0, t4 = tiempo_entre_lineas(tray)
tiempo = t0 - t4

In [ ]:
carril = next(iter(car["laneIndex"]))
distancias = []

for c in gt["cars"]:
    if c["valid"] and len(c["intersections"]) == 2 and c["intersections"][1]["videoTime"] < 1200:
        dt_gt = c["intersections"][1]["videoTime"] - c["intersections"][0]["videoTime"]
        distancias.append(c["speed"] / 3.6 * dt_gt)

D = np.median(distancias)

In [ ]:
velocidad = D / tiempo * 3.6
print(f"mi radar: {velocidad:.1f} km/h | LiDAR: {car['speed']:.1f} km/h | error: {velocidad - car['speed']:+.1f} km/h")

In [ ]:
def calibrar_distancia(gt, t_max=1200):
    distancias = []
    for c in gt["cars"]:
        if c["valid"] and len(c["intersections"]) == 2 and c["intersections"][1]["videoTime"] < t_max:
            dt = c["intersections"][1]["videoTime"] - c["intersections"][0]["videoTime"]
            distancias.append(c["speed"] / 3.6 * dt)
    return np.median(distancias)

In [ ]:
modelo = YOLO("yolo11n.pt")

car = coche_por_id(gt, CLIPS_TRACKING["paralelo"]["carId"])
tInicio, tFinal = tiempoCoche(car["intersections"])
df = detectar_clip(ruta, tInicio - 5.0, tFinal + 1.5, mask)

dibujar_df(ruta, tInicio - 5.0, tFinal + 1.5, df)

!ffmpeg -y -loglevel error -i detecciones.mp4 -c:v libx264 detecciones_h264.mp4
display(Video("detecciones_h264.mp4", embed=True))

In [ ]:
def velocidades_clip(df, gt):
  resultados = []

  D = calibrar_distancia(gt)


  for coche, tray in df[df["id"] >= 0].groupby("id"):
      tray = tray.sort_values("frame")
      cruces = tiempo_entre_lineas(tray, Y_LINEA_4, Y_LINEA_0)

      if cruces is None:
        continue

      t0, t4 = cruces
      tiempo = t0 - t4

      if tiempo is not None and not tiempo == 0.0:
          resultados.append({"id" : coche, "t4" : t4, "tiempo" : tiempo, "velocidad" : D / tiempo * 3.6})

  return pd.DataFrame(resultados)

gt_validos = [c for c in gt["cars"] if c["valid"] and len(c["intersections"]) == 2]

def emparejar(t4, gt_validos, tolerancia=0.2):
    mejor = min(gt_validos, key=lambda c: abs(c["intersections"][0]["videoTime"] - t4))
    if abs(mejor["intersections"][0]["videoTime"] - t4) < tolerancia:
        return mejor
    return None


resultados = velocidades_clip(df, gt)

print(resultados)

In [ ]:
car = coche_por_id(gt, CLIPS_TRACKING["adelantamiento"]["carId"])
tInicio, tFinal = tiempoCoche(car["intersections"])
df = detectar_clip(ruta, tInicio - 3.0, tFinal + 3.0, mask)

dibujar_df(ruta, tInicio - 3.0, tFinal + 3.0, df)

resultados = velocidades_clip(df, gt)
print(resultados)

!ffmpeg -y -loglevel error -i detecciones.mp4 -c:v libx264 detecciones_h264.mp4
display(Video("detecciones_h264.mp4", embed=True))

In [ ]:
def evaluar(res, gt_validos, tolerancia=0.2):
    parejas = [emparejar(t, gt_validos, tolerancia) for t in res["t4"]]
    res = res.copy()
    res["carId_gt"] = [p["carId"] if p else None for p in parejas]
    res["v_gt"]     = [p["speed"] if p else None for p in parejas]
    res["error"]    = res["velocidad"] - res["v_gt"]
    return res

In [ ]:
res = velocidades_clip(df, gt)
print(evaluar(res, gt_validos))

In [ ]:
import os, time, json

SALIDA = "/content/drive/MyDrive/radar-trafico/data/processed/ajuste"
os.makedirs(SALIDA, exist_ok=True)

TROZO, SOLAPE, T_FIN = 60, 5, 1200          # bloque de ajuste: segundos 0–1200

# Configuración del experimento, guardada junto a los datos
config = {"modelo": "yolo11n.pt", "tracker": "bytetrack.yaml (por defecto)",
          "clases": [2, 3, 5, 7], "trozo_s": TROZO, "solape_s": SOLAPE, "t_fin": T_FIN}
with open(f"{SALIDA}/config.json", "w") as f:
    json.dump(config, f, indent=2)

t_total = time.time()
for inicio in range(0, T_FIN, TROZO):
    fichero = f"{SALIDA}/tracks_{inicio:04d}.csv"
    if os.path.exists(fichero):
        print("ya existe, se salta:", fichero)
        continue

    t0 = time.time()
    df_t = detectar_clip(ruta, max(0, inicio - SOLAPE), inicio + TROZO + SOLAPE, mask)
    df_t["trozo"] = inicio
    df_t.to_csv(fichero, index=False)
    print(f"trozo {inicio:4d}-{inicio + TROZO:4d} s | {len(df_t):6d} filas | {time.time() - t0:5.0f} s")

print(f"Total: {(time.time() - t_total) / 60:.1f} min")

In [ ]:
import glob
ficheros = sorted(glob.glob(f"{SALIDA}/tracks_*.csv"))
assert len(ficheros) == 20, f"faltan trozos: solo hay {len(ficheros)}"

tracks = pd.concat([pd.read_csv(f) for f in ficheros], ignore_index=True)
tracks.to_csv(f"{SALIDA}/../tracks_ajuste.csv", index=False)
print(len(tracks), "filas en total")

In [ ]:
tracks = pd.read_csv(f"{SALIDA}/../tracks_ajuste.csv")
D = calibrar_distancia(gt, t_max=1200)

# 1. Velocidades del radar, trozo a trozo
filas = []
for (trozo, coche), tray in tracks[tracks["id"] >= 0].groupby(["trozo", "id"]):
    cruces = tiempo_entre_lineas(tray.sort_values("frame"), Y_LINEA_4, Y_LINEA_0)
    if cruces is None:
        continue
    t0, t4 = cruces
    if t0 > t4 and trozo <= t4 < trozo + TROZO:        # cada coche, solo en su propio trozo
        filas.append({"trozo": trozo, "id": coche, "t4": t4, "tiempo": t0 - t4,
                      "velocidad": D / (t0 - t4) * 3.6})
radar = pd.DataFrame(filas)

# 2. Emparejar con la gt del bloque de ajuste
gt_ajuste = [c for c in gt["cars"] if c["valid"] and len(c["intersections"]) == 2
             and c["intersections"][0]["videoTime"] < 1200]
ev = evaluar(radar, gt_ajuste)

# 3. Cifras
emp = ev.dropna(subset=["carId_gt"])
print(f"Coches medidos por el radar:      {len(radar)}")
print(f"  emparejados con la gt:          {len(emp)}")
print(f"  sin pareja en la gt:            {len(radar) - len(emp)}")
print(f"  coches gt con 2+ IDs del radar: {emp['carId_gt'].duplicated().sum()}")
print(f"Cobertura: {emp['carId_gt'].nunique()} de {len(gt_ajuste)} coches válidos de la gt "
      f"({emp['carId_gt'].nunique() / len(gt_ajuste):.1%})")
print(f"Error medio absoluto: {emp['error'].abs().mean():.2f} km/h")
print(f"Error medio con signo: {emp['error'].mean():+.2f} km/h")
print(f"Coches con |error| < 3 km/h: {(emp['error'].abs() < 3).mean():.1%}")